# SENTRi-X · ToN-IoT baseline

**Start here.** This notebook prepares one dataset, trains a Random Forest and a real
1D CNN, compares RF / CNN / Hybrid on the same holdout, and saves one candidate package.
The earlier ToN notebooks are preserved in `archive/ton_iot/` for reference.

Select your project's **TensorFlow Python environment**, restart the kernel, and run
from top to bottom. Edit only the configuration cell for your first run.
The default `smoke` run checks the workflow; its scores are **not thesis results**.
Raw CSV files must already be in `data/raw/ton_iot/`.

### Protocol for this baseline

- Binary target: **0 = Benign, 1 = Attack**; canonical 28 input features.
- One seeded uniform reservoir sample across the selected source CSV files.
- Retain distinct source records and their original labels. Keep duplicate/session
  groups entirely within one partition, including groups with conflicting labels.
- Reserve 20% of groups for testing; reserve 10% of the remaining groups for validation.
  This targets about **72% training / 8% validation / 20% testing**. Group sizes can
  change the actual row percentages; the notebook reports those percentages.
- Fit the scaler on original training rows, then oversample **training only**.
- Fixed RF/CNN settings, equal fusion `(p_rf + p_cnn) / 2`, threshold **0.5**.
  Test results do not select settings, epochs, weights, seeds or thresholds.
- Report empirical performance on the **sampled, group-disjoint holdout**. Ingestion
  weights do not establish final holdout population weights. This experiment alone
  does not demonstrate performance on Pi traffic or on the other datasets.

## 1. Locate the project and import the shared functions

The notebook is your working interface. The imported functions keep feature preparation,
splitting, prediction and saved artifacts consistent with the application and CLI.
No earlier notebook or old processed `.pkl` file is required.

In [ ]:
import argparse
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT_DIR = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sentrix_ml").is_dir()), None
)
if ROOT_DIR is None:
    raise RuntimeError("Open this notebook from your SENTRi-X repository or its notebooks folder.")
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sentrix_ml.adapters.ton_iot import load_ton_iot
from sentrix_ml.train_source import split_source_data, save_source_candidate, SOURCE_PROTOCOL, SOURCE_EVALUATION_SPLIT
from sentrix_ml.preprocessing import PreprocessingPipeline
from sentrix_ml.balancing import balance_training_rows
from sentrix_ml.training import train_rf, train_cnn
from sentrix_ml.inference import hybrid_predict
from sentrix_ml.evaluation import compute_multimode_metrics
from sentrix_ml.packaging import file_sha256, validate_package
from sentrix_ml.schema import EXPECTED_FEATURES

print("Project:", ROOT_DIR)

## 2. Configure this run

Begin with `RUN_TYPE = "smoke"`. It still trains both real models, using fewer rows,
trees and epochs. After that works on your laptop, a **fresh run** with `RUN_TYPE = "full"`
uses the declared baseline settings below. Do not change settings in response to test
scores. Each run gets a new output directory; existing candidates are preserved.

`max_files=None` reads all matching source files. `sample_n` bounds the selected rows,
not the number of source rows read.

In [ ]:
RUN_TYPE = "smoke"  # Choose "smoke" for verification, "full" for a declared research run.
if RUN_TYPE not in {"smoke", "full"}:
    raise ValueError("RUN_TYPE must be 'smoke' or 'full'")

RUN_ID = f"ton_{RUN_TYPE}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S_%fZ}"
RUN_DIR = ROOT_DIR / "outputs" / "notebook_runs" / RUN_ID
args = argparse.Namespace(
    data_dir=str(ROOT_DIR / "data" / "raw" / "ton_iot"),
    max_files=None,
    sample_n=2000 if RUN_TYPE == "smoke" else 50000,
    test_fraction=0.20,
    val_fraction=0.10,
    rf_estimators=5 if RUN_TYPE == "smoke" else 100,
    rf_depth=None,
    cnn_epochs=1 if RUN_TYPE == "smoke" else 10,
    batch_size=64 if RUN_TYPE == "smoke" else 256,
    seed=42,
    run_type=RUN_TYPE,
    output_dir=str(ROOT_DIR / "models" / "candidates" / RUN_ID),
)

In [ ]:
# These defaults limit CPU thread use in a notebook; set before importing TensorFlow.
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
try:
    import tensorflow as tf
except ImportError as exc:
    raise RuntimeError("Select your project's TensorFlow kernel. A placeholder CNN is not used.") from exc

if not Path(args.data_dir).exists():
    raise FileNotFoundError(f"Raw ToN-IoT data was not found at {args.data_dir}. Edit data_dir above.")
if Path(args.output_dir).exists() and any(Path(args.output_dir).iterdir()):
    raise FileExistsError("Candidate output already contains files. Restart with a new RUN_ID.")
RUN_DIR.mkdir(parents=True, exist_ok=False)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(args.seed)
args.notebook_sha256 = file_sha256(ROOT_DIR / "notebooks" / "01_ToN_IoT_Baseline.ipynb")
config = dict(vars(args), protocol=SOURCE_PROTOCOL, evaluation_split=SOURCE_EVALUATION_SPLIT)
(RUN_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
display(pd.Series(config, name="Configured value").to_frame())
print("TensorFlow:", tf.__version__)
print("SMOKE CHECK ONLY — not thesis results" if args.run_type == "smoke" else "Full configured baseline run")

## 3. Load and inspect the raw data

Identifiers and source metadata are kept separately from model inputs. The adapter
validates required measurements and records exclusions; some optional fields may be
imputed. Compare source and selected class counts before interpreting results.

In [ ]:
X, y, info = load_ton_iot(
    args.data_dir, max_files=args.max_files, sample_n=args.sample_n, seed=args.seed,
)
assert list(X.columns) == EXPECTED_FEATURES
assert X.index.equals(y.index) and X.index.equals(info["metadata"].index)
assert np.isfinite(X.to_numpy()).all()
display(pd.DataFrame({
    "Valid source rows": info["source_class_counts"],
    "Selected rows": info["selected_class_counts"],
}).fillna(0).astype(int).rename(index={"0": "Benign", "1": "Attack"}))
print("Files:", info["files_considered"])
print("Raw rows read:", info["total_rows_considered"])
print("Cleaning exclusions:", info["exclusion_reasons"])
display(X.head())

## 4. Create and freeze the shared partitions

Both models use these exact partitions. Common label profiles are stratified by group;
singleton profiles use one seeded allocation. The split stops if the actual partitions
lack the required benign/attack support. It never deletes conflicting labels or retries
seeds to improve results. Source file hashes and exact row identities are saved here.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, split_manifest = split_source_data(
    X, y, info, test_fraction=args.test_fraction,
    val_fraction=args.val_fraction, seed=args.seed,
)
group_sets = [set(getattr(split_manifest, p + "_group_ids")) for p in ("train", "val", "test")]
assert all(group_sets[i].isdisjoint(group_sets[j]) for i, j in ((0, 1), (0, 2), (1, 2)))
assert sum(map(len, (X_train, X_val, X_test))) + split_manifest.excluded_rows == len(X)

partition_table = pd.DataFrame([
    {"Partition": name, "Rows": len(labels), "Benign": int((labels == 0).sum()),
     "Attack": int((labels == 1).sum()), "Percent of selected rows": 100 * len(labels) / len(X)}
    for name, labels in (("Training", y_train), ("Validation", y_val), ("Test", y_test))
]).set_index("Partition")
display(partition_table)
partition_table.to_csv(RUN_DIR / "partition_counts.csv")
split_manifest.save(RUN_DIR / "split_manifest.json")
print("Split saved. Group overlap: 0. Record policy:", split_manifest.duplicate_group_policy)

## 5. Fit preprocessing and balance training rows

The scaler sees only the original training partition. Random oversampling then repeats
minority training rows; it never creates synthetic validation or test observations.
RF and CNN receive the same resulting fitting rows.

In [ ]:
pipeline = PreprocessingPipeline().fit(X_train)
X_train_scaled = pipeline.transform(X_train)
X_val_scaled = pipeline.transform(X_val)
X_test_scaled = pipeline.transform(X_test)
X_fit, y_fit, balancing_audit = balance_training_rows(X_train_scaled, y_train, seed=args.seed)
np.testing.assert_allclose(pipeline.scaler.mean_, X_train.to_numpy().mean(axis=0))
print("Original training rows:", len(y_train), "| Fitting rows after oversampling:", len(y_fit))
print("Validation rows:", len(y_val), "| Test rows:", len(y_test))
display(pd.DataFrame({
    "Original training": balancing_audit["original_class_counts"],
    "After oversampling": balancing_audit["resampled_class_counts"],
}).rename(index={"0": "Benign", "1": "Attack"}))

## 6. Train the Random Forest

Training uses the fixed settings from the configuration cell. Test labels are not used
for fitting or model selection.

In [ ]:
rf_model = train_rf(
    X_fit, y_fit, n_estimators=args.rf_estimators,
    max_depth=args.rf_depth, random_state=args.seed,
)
print("RF fitted. Classes:", rf_model.classes_, "| Trees:", rf_model.n_estimators)

## 7. Train the 1D CNN

The CNN takes `(rows, 28, 1)` inputs and outputs one attack probability per row. Its
validation curves use the separate validation partition. The configured epoch count
is fixed in advance; the holdout is evaluated after training.

In [ ]:
cnn_model, history = train_cnn(
    X_fit, y_fit, X_val_scaled, y_val,
    epochs=args.cnn_epochs, batch_size=args.batch_size, verbose=1,
)
history_table = pd.DataFrame(history.history)
history_table.index = np.arange(1, len(history_table) + 1)
history_table.index.name = "Epoch"
history_table.to_csv(RUN_DIR / "cnn_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, metric, title in zip(axes, ("loss", "accuracy"), ("CNN loss", "CNN accuracy")):
    ax.plot(history_table.index, history_table[metric], marker="o", label="Training")
    ax.plot(history_table.index, history_table["val_" + metric], marker="o", label="Validation")
    ax.set(xlabel="Epoch", ylabel=metric.title(), title=title)
    ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "cnn_training.png", dpi=160)
plt.show()

## 8. Evaluate RF, CNN and Hybrid on the same holdout

The decision rule is **Attack when probability ≥ 0.5**. Hybrid uses the arithmetic mean
of the two attack probabilities. These are empirical, unweighted holdout metrics.
The confusion matrices show how many benign flows were falsely flagged and how many
attack flows were missed. Smoke-run scores are only execution checks.

In [ ]:
y_pred, p_hybrid, p_rf, p_cnn = hybrid_predict(
    X_test_scaled, rf_model=rf_model, cnn_model=cnn_model, mode="hybrid",
)
evaluation = compute_multimode_metrics(
    y_test.to_numpy(dtype=int), p_rf=p_rf, p_cnn=p_cnn, p_hybrid=p_hybrid,
    domain="ton_iot", dataset="ton_iot", run_type=args.run_type,
    evaluation_split=SOURCE_EVALUATION_SPLIT,
)
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc", "specificity", "false_positive_rate"]
metrics_table = pd.DataFrame({m: {k: v[k] for k in metric_names} for m, v in evaluation.modes.items()}).T
metrics_table.index.name = "Mode"
display(metrics_table)
metrics_table.to_csv(RUN_DIR / "holdout_metrics.csv")

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, mode in zip(axes, ("rf", "cnn", "hybrid")):
    cm = evaluation.modes[mode]["confusion_matrix"]
    ConfusionMatrixDisplay(np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]]),
                           display_labels=["Benign", "Attack"]).plot(ax=ax, colorbar=False)
    ax.set_title(mode.upper())
fig.suptitle(f"{args.run_type.upper()} · sampled ToN-IoT holdout")
fig.tight_layout()
fig.savefig(RUN_DIR / "holdout_confusion_matrices.png", dpi=160)
plt.show()

## 9. Save and verify the candidate

The package contains both models, the fitted scaler, the frozen split, metrics and
per-row prediction evidence, with artifact hashes linking them together. The evidence
CSV includes original source file/row references. Its inclusion probabilities and
weights describe **ingestion**, not final test-set selection, and do not weight the
reported metrics.

The package stays in `models/candidates/`. A smoke package cannot be activated for
production. This notebook does not switch dashboard models.

In [ ]:
package_manifest = save_source_candidate(
    args=args, rf_model=rf_model, cnn_model=cnn_model, pipeline=pipeline,
    split_manifest=split_manifest, X_test=X_test, y_test=y_test,
    p_rf=p_rf, p_cnn=p_cnn, p_hybrid=p_hybrid, balancing_audit=balancing_audit,
)
validate_package(args.output_dir, strict_deployable=(args.run_type == "full"))
assert package_manifest.is_mock is False
print("Candidate:", Path(args.output_dir))
print("Tables and figures:", RUN_DIR)
print("Run type:", package_manifest.evaluation_run_type)
print("Complete. No dashboard model was activated.")

## What to keep and what comes next

Keep the candidate directory, this run's tables/figures, and the configuration together.
For a thesis baseline, report only the actual full-run results and their saved protocol.
The source CSV files are required to reproduce source-row evidence.

This rebuild covers **ToN-IoT baseline training**. The next notebook work is explanation
of this exact candidate, then BoT-IoT / CIC-IDS2017 adaptation and Omni. Their previous
sampling/integration findings remain open. Pi feature parity, attack-script/Snort
comparison and expert evaluation are separate checks; this notebook does not establish
them. Earlier notebooks in the archive may depend on obsolete data and paths.